# Configured oxygen and water-activity modifiers example

This notebook is a public configured-workflow example for the explicit `oxygen_monod` and `water_activity_threshold` rate modifiers. It uses artificial framework-benchmark temp config data created in a temporary notebook output folder and makes no biological claim.

Guardrails: this is not an empirical validation, calibration, literature comparison, organism-specific physiology, oxygen/redox model, or water-binding model example. It adds no oxygen consumption state, gas transfer, redox balance, anaerobic metabolism, substrate water-binding model, fitted oxygen/water-activity response curve, no inferred environment response, no EnvironmentGrid behavior change, hidden notebook science, or thermodynamic enforcement. The notebook uses package APIs and configured workflow outputs only; it does not define rate laws, solver logic, or hidden notebook science.


In [ ]:
import csv
import json
import os
import sys
from copy import deepcopy
from pathlib import Path

import yaml

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import run_configured_model

SOURCE_CONFIG = ROOT / "data" / "model_configs" / "toy_homogeneous_ab.yml"
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "notebooks" / "examples" / "Outputs")))
OUTPUT = OUTPUT_ROOT / "18_configured_oxygen_water_modifiers_example"
BASE_OUTPUT = OUTPUT / "base"
MODIFIED_OUTPUT = OUTPUT / "modified"
CONFIG = OUTPUT / "configured_inputs" / "toy_homogeneous_oxygen_water_modifiers.yml"


## Build an explicit temporary config

The source config is the existing homogeneous software benchmark. The notebook adds explicit artificial parameter records and explicit configured modifier declarations. The package runner still owns model assembly, parameter checking, environment reading, process rates, assumptions, and configured output writing.

The environment comes from the configured environment entity, so oxygen concentration and water activity are explicit input values. If the oxygen/water-activity values, oxygen units, or required parameter records are removed, the configured workflow fails rather than using fallback constants.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG.parent.mkdir(parents=True, exist_ok=True)

source = "FungMod configured oxygen-water modifier software benchmark."
config = yaml.safe_load(SOURCE_CONFIG.read_text(encoding="utf-8"))
config = deepcopy(config)
config["name"] = "toy homogeneous explicit oxygen and water-activity modifiers benchmark"
config["mode"] = "toy"
config["maturity"] = "framework_benchmark"
config["provenance"] = {
    "source": source,
    "measurement_method": "defined software benchmark",
    "confidence_level": "testing",
    "notes": "Artificial configured-workflow demo; not biological oxygen or water-activity response evidence.",
    "validity_range": "framework tests and public examples only",
    "units": "not_applicable",
}

config["parameters"][0]["parameters"].extend(
    [
        {
            "name": "toy oxygen half saturation",
            "symbol": "K_O2_env",
            "value": 0.25,
            "units": "mole / liter",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not fitted oxygen physiology or redox behavior.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
        {
            "name": "toy minimum water activity",
            "symbol": "a_w_min_env",
            "value": 0.9,
            "units": "dimensionless",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not a fitted moisture response or substrate water-binding model.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
    ]
)
config["processes"][0]["modifiers"] = [
    {
        "type": "oxygen_monod",
        "half_saturation_symbol": "K_O2_env",
        "oxygen_units": "mole / liter",
        "source": source,
    },
    {
        "type": "water_activity_threshold",
        "minimum_water_activity_symbol": "a_w_min_env",
        "source": source,
    },
]

CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
{
    "temporary_config": str(CONFIG),
    "modifier_types": [item["type"] for item in config["processes"][0]["modifiers"]],
    "explicit_parameter_symbols": [item["symbol"] for item in config["parameters"][0]["parameters"] if item["symbol"].endswith("_env")],
    "configured_environment_path": config["entities"]["environment"]["path"],
}


## Run the package workflow and inspect emitted outputs

`run_configured_model(...)` validates the explicit symbols and oxygen units, reads the configured environment, builds package process modifiers, and writes the configured output bundle. The comparison to the unmodified benchmark is only a software smoke check that the configured modifiers are active; it is not validation, calibration, empirical comparison, an inferred oxygen response, or an inferred water-activity response curve.


In [ ]:
base_result = run_configured_model(SOURCE_CONFIG, output_dir=BASE_OUTPUT)
modified_result = run_configured_model(CONFIG, output_dir=MODIFIED_OUTPUT)

metadata = json.loads((MODIFIED_OUTPUT / "configured_metadata.json").read_text(encoding="utf-8"))
input_config = json.loads((MODIFIED_OUTPUT / "input_model_config.json").read_text(encoding="utf-8"))
merged_parameters = json.loads((MODIFIED_OUTPUT / "merged_parameters.json").read_text(encoding="utf-8"))
assumptions = json.loads((MODIFIED_OUTPUT / "assumptions.json").read_text(encoding="utf-8"))
entity_index = json.loads((MODIFIED_OUTPUT / "entity_snapshots" / "index.json").read_text(encoding="utf-8"))
environment_entry = next(entry for entry in entity_index["entities"] if entry["role"] == "environment")
environment = json.loads((MODIFIED_OUTPUT / environment_entry["snapshot_path"]).read_text(encoding="utf-8"))

with (BASE_OUTPUT / "process_rates.csv").open(newline="", encoding="utf-8") as handle:
    base_rates = list(csv.DictReader(handle))
with (MODIFIED_OUTPUT / "process_rates.csv").open(newline="", encoding="utf-8") as handle:
    modified_rates = list(csv.DictReader(handle))

summary = {
    "base_first_rate": float(base_rates[0]["value"]),
    "modified_first_rate": float(modified_rates[0]["value"]),
    "process_rates_changed_by_configured_modifiers": float(modified_rates[0]["value"]) != float(base_rates[0]["value"]),
    "configured_modifier_types": [row["type"] for row in metadata["configured_process_modifiers"]],
    "configured_environment_values": {
        "oxygen_concentration": environment["oxygen_concentration"],
        "water_activity": environment["water_activity"],
    },
    "explicit_parameter_symbols": [item["symbol"] for item in merged_parameters["parameters"] if item["symbol"].endswith("_env")],
    "assumption_names": [item["name"] for item in assumptions if "oxygen" in item["name"] or "water" in item["name"]],
    "modifier_limitations": [row["limitation"] for row in metadata["configured_process_modifiers"]],
    "input_modifier_declarations": input_config["processes"][0]["modifiers"],
}

assert modified_result.solver_metadata["success"] is True
assert "a_to_b" in base_result.process_rates
assert "a_to_b" in modified_result.process_rates
assert summary["process_rates_changed_by_configured_modifiers"]
assert {"K_O2_env", "a_w_min_env"}.issubset(summary["explicit_parameter_symbols"])
assert summary["configured_environment_values"]["oxygen_concentration"]["value"] == 0.25
assert summary["configured_environment_values"]["water_activity"]["value"] == 0.98
summary


## What this proves and what it does not prove

This example proves that configured generic processes can opt into the package's existing Monod oxygen and binary water-activity threshold modifiers when the modifier parameter records, oxygen units, and environment oxygen/water-activity values are explicit. It also shows where the configured workflow records those assumptions, limitations, merged parameters, environment/entity snapshots, process rates, and modifier metadata.

It does not add oxygen consumption state, gas transfer, redox balance, anaerobic metabolism, substrate water-binding model, a fitted oxygen/water-activity response curve, validation data, calibration, empirical comparison, inferred environment response, `EnvironmentGrid` behavior change, hidden notebook science, thermodynamic enforcement, or solver/model behavior changes.
